# Question 2 — Exercise A.2 (Convoy)

Ram + 3 wagons. Key <b>r</b> = wheels cube/cylinder/sphere.

### Colors kaise change karein (bahut easy)
Har shape aise banti hai:
```
new Cube(this.gl, [R1,G1,B1], [R2,G2,B2])
```
- pehla array = left color  
- doosra array = right color (gradient)  
- values **0 se 1** (example red = `[1, 0, 0]`, gold = `[0.95, 0.75, 0.2]`)

Background:
```
this.gl.clearColor(R, G, B, 1);
```



In [ ]:
%%html

<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/cgraobject.js"></script>
<script src="https://git.io/glm-js.min.js"></script>


<script id="my-vertex-shader" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
varying vec3 ex_Color;
void main(void){
  gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);
  ex_Color = in_Color;
}
</script>


<script id="my-fragment-shader" type="x-shader/x-fragment">
precision mediump float;
varying vec3 ex_Color;
void main(void){ gl_FragColor = vec4(ex_Color, 1.0); }
</script>


<div id="cgraStatus" style="font:13px/1.4 system-ui,sans-serif;color:#0a7;min-height:1.2em;"></div>
<script>
function cgraReady(){
  return typeof DEECapp!=='undefined' && typeof DEECshader!=='undefined'
      && typeof CGRAobject!=='undefined' && typeof glm!=='undefined';
}
function cgraBoot(fn){
  var n=0;
  (function wait(){
    n++;
    if(!cgraReady()){
      if(n===30){ var e=document.getElementById('cgraStatus'); if(e){ e.style.color='#c60'; e.textContent='Waiting libraries...'; } }
      if(n<250) return setTimeout(wait,40);
      var e2=document.getElementById('cgraStatus'); if(e2){ e2.style.color='#c00'; e2.textContent='ERROR: libs not loaded (F12)'; }
      return;
    }
    var e3=document.getElementById('cgraStatus'); if(e3) e3.textContent='';
    try{ fn(); }catch(err){ console.error(err); alert('Error: '+err.message); }
  })();
}
</script>

<div style="font:14px/1.45 system-ui,sans-serif;color:#243;margin:0 0 8px 0;max-width:720px;">
  <b>Q2 Convoy</b> — press <b>r</b> to change wheels<br>
  <span style="color:#666;">Voilà only (not Run). Click canvas before keys.</span>
</div>
<canvas id="cQ2" width="720" height="460"
  style="border:1px solid #2a3340;border-radius:8px;background:#1a2230;display:block;box-shadow:0 8px 24px rgba(0,0,0,.18);"></canvas>
<script>
cgraBoot(function(){
// ============================================================
// CGRA A1 — BASIC shapes (all extend CGRAobject)
// Keep this simple for oral defense.
// Color: smooth gradient from color1 to color2 along X.
// ============================================================

function mixColor(c1, c2, t) {
    // t from 0 to 1
    return [
        c1[0] * (1 - t) + c2[0] * t,
        c1[1] * (1 - t) + c2[1] * t,
        c1[2] * (1 - t) + c2[2] * t
    ];
}

function pushTri(verts, cols, ax, ay, az, bx, by, bz, cx, cy, cz, c1, c2) {
    // one triangle = 3 vertices
    var pts = [[ax, ay, az], [bx, by, bz], [cx, cy, cz]];
    for (var i = 0; i < 3; i++) {
        verts.push(pts[i][0], pts[i][1], pts[i][2]);
        // gradient by X: -1..1 mapped to 0..1
        var t = (pts[i][0] + 1.0) * 0.5;
        if (t < 0) t = 0;
        if (t > 1) t = 1;
        var col = mixColor(c1, c2, t);
        cols.push(col[0], col[1], col[2]);
    }
}

function uploadBuffers(obj, verts, cols) {
    obj.numvertices = verts.length / 3;
    obj.vertexbuffer = obj.gl.createBuffer();
    obj.gl.bindBuffer(obj.gl.ARRAY_BUFFER, obj.vertexbuffer);
    obj.gl.bufferData(obj.gl.ARRAY_BUFFER, new Float32Array(verts), obj.gl.STATIC_DRAW);

    obj.colorbuffer = obj.gl.createBuffer();
    obj.gl.bindBuffer(obj.gl.ARRAY_BUFFER, obj.colorbuffer);
    obj.gl.bufferData(obj.gl.ARRAY_BUFFER, new Float32Array(cols), obj.gl.STATIC_DRAW);
}

// ---------- CUBE: 6 faces * 2 triangles * 3 verts = 36 ----------
class Cube extends CGRAobject {
    constructor(glcontext, color1, color2) {
        super(glcontext);
        var c1 = color1 || [1, 0, 0];
        var c2 = color2 || [0, 0, 1];
        var verts = [];
        var cols = [];

        // unit cube corners from -1 to 1
        // faces: front, back, left, right, top, bottom
        // each face = 2 triangles
        var f = [
            // front z=+1
            [-1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, 1, -1, 1, 1],
            // back z=-1
            [1, -1, -1, -1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, 1, -1],
            // left x=-1
            [-1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, 1, -1],
            // right x=+1
            [1, -1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1],
            // top y=+1
            [-1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1, 1, -1, -1, 1, -1],
            // bottom y=-1
            [-1, -1, -1, 1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, -1, -1, 1]
        ];

        for (var fi = 0; fi < f.length; fi++) {
            var a = f[fi];
            // two triangles already listed as 6 verts
            pushTri(verts, cols, a[0], a[1], a[2], a[3], a[4], a[5], a[6], a[7], a[8], c1, c2);
            pushTri(verts, cols, a[9], a[10], a[11], a[12], a[13], a[14], a[15], a[16], a[17], c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- DISC: center + ring (>=10) ----------
class Disc extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [1, 1, 0];
        var c2 = color2 || [1, 0.5, 0];
        n = n || 12; // at least 10
        var verts = [];
        var cols = [];
        var r = 1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // triangle: center, p0, p1  (in XZ plane, y=0)
            pushTri(verts, cols, 0, 0, 0, x0, 0, z0, x1, 0, z1, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- CONE: apex + base circle (>=10) ----------
class Cone extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [0.8, 0.4, 0.1];
        var c2 = color2 || [0.3, 0.15, 0.05];
        n = n || 12;
        var verts = [];
        var cols = [];
        var r = 1.0;
        var apexY = 1.0;
        var baseY = -1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // side triangle
            pushTri(verts, cols, 0, apexY, 0, x0, baseY, z0, x1, baseY, z1, c1, c2);
            // base triangle (optional close)
            pushTri(verts, cols, 0, baseY, 0, x1, baseY, z1, x0, baseY, z0, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- CYLINDER: two circles (>=10), quads -> 2 tris ----------
class Cylinder extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [0.6, 0.6, 0.6];
        var c2 = color2 || [0.2, 0.2, 0.2];
        n = n || 12;
        var verts = [];
        var cols = [];
        var r = 1.0;
        var yTop = 1.0, yBot = -1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // side: two triangles
            pushTri(verts, cols, x0, yBot, z0, x1, yBot, z1, x1, yTop, z1, c1, c2);
            pushTri(verts, cols, x0, yBot, z0, x1, yTop, z1, x0, yTop, z0, c1, c2);
            // top cap
            pushTri(verts, cols, 0, yTop, 0, x0, yTop, z0, x1, yTop, z1, c1, c2);
            // bottom cap
            pushTri(verts, cols, 0, yBot, 0, x1, yBot, z1, x0, yBot, z0, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- SPHERE: latitude strips (>=10) * rings (>=10) ----------
class Sphere extends CGRAobject {
    constructor(glcontext, color1, color2, stacks, slices) {
        super(glcontext);
        var c1 = color1 || [0.2, 0.6, 1.0];
        var c2 = color2 || [1.0, 0.2, 0.4];
        stacks = stacks || 12; // latitude strips
        slices = slices || 12; // vertices per ring
        var verts = [];
        var cols = [];
        var R = 1.0;

        for (var i = 0; i < stacks; i++) {
            // elevation from -PI/2 to +PI/2
            var lat0 = -Math.PI / 2 + (i / stacks) * Math.PI;
            var lat1 = -Math.PI / 2 + ((i + 1) / stacks) * Math.PI;
            var y0 = R * Math.sin(lat0);
            var y1 = R * Math.sin(lat1);
            var r0 = R * Math.cos(lat0);
            var r1 = R * Math.cos(lat1);

            for (var j = 0; j < slices; j++) {
                var a0 = (j / slices) * 2 * Math.PI;
                var a1 = ((j + 1) / slices) * 2 * Math.PI;

                var x00 = r0 * Math.cos(a0), z00 = r0 * Math.sin(a0);
                var x01 = r0 * Math.cos(a1), z01 = r0 * Math.sin(a1);
                var x10 = r1 * Math.cos(a0), z10 = r1 * Math.sin(a0);
                var x11 = r1 * Math.cos(a1), z11 = r1 * Math.sin(a1);

                // two triangles between rings
                pushTri(verts, cols, x00, y0, z00, x01, y0, z01, x11, y1, z11, c1, c2);
                pushTri(verts, cols, x00, y0, z00, x11, y1, z11, x10, y1, z10, c1, c2);
            }
        }
        uploadBuffers(this, verts, cols);
    }
}

function T(x,y,z){ return glm.translate(glm.mat4(1.0), glm.vec3(x,y,z)); }
function Rdeg(deg,x,y,z){ return glm.rotate(glm.mat4(1.0), glm.radians(deg), glm.vec3(x,y,z)); }
function Su(s){ return glm.mat4(glm.mat3(s)); }
function mul2(a,b){ return a['*'](b); }
function mul3(a,b,c){ return a['*'](b['*'](c)); }
function mul4(a,b,c,d){ return a['*'](b['*'](c['*'](d))); }
function makeShader(app){
  var fs=document.getElementById('my-fragment-shader').text;
  var vs=document.getElementById('my-vertex-shader').text;
  app.shaderprog=new DEECshader(app.gl);
  app.shaderprog.srcShaders(vs,fs);
  app.gl.enable(app.gl.DEPTH_TEST);
}
function setSh(obj, app){ obj.setShader(app.shaderprog); }
function draw(obj, M){ obj.setModelTransformation(M); obj.drawit(); }
function drawVP(obj, M, view, proj){ obj.setModelTransformation(M); obj.drawit(view, proj); }

class myapp extends DEECapp{
  counter=0; wheelMode=0;
  initialize(){
    makeShader(this);
    this.gl.clearColor(0.55, 0.72, 0.88, 1);
    this.ground=new Disc(this.gl,[0.42,0.62,0.35],[0.28,0.48,0.25],28);
    this.wood=new Cube(this.gl,[0.72,0.48,0.22],[0.40,0.22,0.08]);      // warm oak
    this.metal=new Cube(this.gl,[0.55,0.60,0.70],[0.25,0.28,0.35]);     // steel
    this.ram=new Cone(this.gl,[0.95,0.75,0.25],[0.75,0.35,0.10],12);     // brass tip
    this.cyl=new Cylinder(this.gl,[0.35,0.40,0.48],[0.15,0.18,0.22],12);
    this.sph=new Sphere(this.gl,[0.85,0.55,0.25],[0.45,0.22,0.10],12,12); // bronze
    this.loadR=new Cube(this.gl,[0.85,0.25,0.25],[0.45,0.10,0.10]);
    this.loadG=new Cube(this.gl,[0.25,0.70,0.40],[0.10,0.40,0.20]);
    this.loadB=new Sphere(this.gl,[0.35,0.55,0.95],[0.15,0.25,0.70],10,10);
    var a=[this.ground,this.wood,this.metal,this.ram,this.cyl,this.sph,this.loadR,this.loadG,this.loadB];
    for(var i=0;i<a.length;i++) setSh(a[i],this);
    window.addEventListener('keydown',(e)=>{
      if(e.key.toLowerCase()==='r'){ this.wheelMode=(this.wheelMode+1)%3; console.log('wheel',this.wheelMode); }
    },false);
  }
  wheel(M,x,y){
    var Wm=mul3(M, T(x,y,0), Su(0.045));
    if(this.wheelMode===0) draw(this.metal,Wm);
    else if(this.wheelMode===1) draw(this.cyl,Wm);
    else draw(this.sph,Wm);
  }
  vehicle(M, i){
    var s = (i===0)? 0.13 : (0.09+0.012*i);
    draw(this.wood, mul2(M, Su(s)));
    if(i===0) draw(this.ram, mul3(M, T(0.20,0,0), mul2(Rdeg(90,0,0,1), Su(0.07))));
    if(i===1) draw(this.loadR, mul3(M, T(0,0.13,0), Su(0.045)));
    if(i===2) draw(this.loadG, mul3(M, T(0,0.13,0), Su(0.045)));
    if(i===3) draw(this.loadB, mul3(M, T(0,0.14,0), Su(0.05)));
    this.wheel(M,-0.09,-0.09); this.wheel(M,0.09,-0.09);
    this.wheel(M,-0.09, 0.09); this.wheel(M,0.09, 0.09);
  }
  render(){
    this.gl.clear(this.gl.COLOR_BUFFER_BIT|this.gl.DEPTH_BUFFER_BIT);
    draw(this.ground, mul3(T(0,-0.38,0), Rdeg(90,1,0,0), Su(1.2)));
    // clean row, slight tilt for depth (no crazy spin)
    var base=Rdeg(18,1,0,0);
    var xs=[0.58, 0.18, -0.22, -0.62];
    for(var i=0;i<4;i++){
      var M=mul3(base, T(xs[i], -0.05, 0), Rdeg(8,0,1,0));
      this.vehicle(M,i);
    }
  }
}
new myapp('cQ2').run();

});
</script>

